In [1]:
import os
import re
import sys

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from company_name_matcher import CompanyNameMatcher  # pip install company-name-matcher

ROOT_PATH = r'D:\Users\wangy\PycharmProjects\GlassDoor'

if ROOT_PATH not in sys.path:
    sys.path.append(ROOT_PATH)

from Constants import Constants as const

In [2]:
glassdoor_path = r'D:\Users\wangy\Documents\data\glassdoor'

In [3]:
gd_df = pd.read_csv(os.path.join(glassdoor_path, 'Glassdoor岗位评价.csv'), nrows=1000)

In [8]:
gd_df.columns = [
    'GD_JobTitle', 'GD_CompanyLink', 'GD_CompanyName', 'GD_CompanyID', 'GD_ReviewDate',
    'GD_Rating', 'GD_ReviewTitle', 'GD_ReviewerStatus', 'GD_Pros', 'GD_Cons', 'GD_Advice',
    'GD_Recommend', 'GD_CEOSupport', 'GD_Outlook', 'GD_CareerOpp', 'GD_CompBenefits',
    'GD_Management', 'GD_WorkLife', 'GD_CultureValues', 'GD_Diversity', 'GD_Index'
]


In [10]:
len(gd_df['GD_JobTitle'].unique())

391

In [11]:
gd_df['GD_JobTitle'].unique()

array([' Manager Design', ' Anonymous Employee', ' Production Engineer',
       ' Senior Account Executive', ' Assistant Manager',
       ' Floor Supervisor', ' Account Manager', ' Marketing',
       ' Security Guard', ' Usher/Ticket Taker',
       ' Customer Service Representative', ' Account Executive',
       ' Information Technologist', ' Usher', ' Cashier', ' Bartender',
       ' Specialty Sales Associate', ' Instructor Pilot', ' Supervisor',
       ' Internship', ' Marketing Coordinator', ' Events Coordinator',
       ' Server/Waiter', ' Beer Vendor', ' Warehouse Associate',
       ' Server/Bartender', ' Catering', ' Concession Cashier', ' ',
       ' Day Warehouse Stock Clerk', ' Sales Intern', ' Security',
       ' Cash Office', ' Caterer', ' Manager', ' Hockey Analyst',
       ' Corporate Partnership', ' Marketing Associate',
       ' Senior Product Designer', ' Copywriter', ' Junior Copywriter',
       ' Marketing Specialist', ' Freelance Graphic Designer',
       ' Product A

In [13]:
gd_df.to_pickle(os.path.join(const.TEMP_PATH, '2008_2023_raw_glassdoor_reviews.pkl'))

In [15]:
gd_df[gd_df['GD_CompanyID'] == 8131102]

,GD_JobTitle,GD_CompanyLink,GD_CompanyName,GD_CompanyID,GD_ReviewDate,GD_Rating,GD_ReviewTitle,GD_ReviewerStatus,GD_Pros,GD_Cons,...,GD_Recommend,GD_CEOSupport,GD_Outlook,GD_CareerOpp,GD_CompBenefits,GD_Management,GD_WorkLife,GD_CultureValues,GD_Diversity,GD_Index
1692809,Marketing and Sales Associate,Reviews/Werner-Elektrik-Reviews-E8131102.htm,Werner Elektrik,8131102,2022-12-08,4.0,Good experience,Current Employee,Flexible working hours Work from home,Low salary Inter team communication,...,v,o,r,4.0,3.0,4.0,3.0,3.0,5.0,NaN


In [16]:
mapping_df = pd.read_csv(os.path.join(glassdoor_path, 'wxr_data', 'glassdoor_to_wrds_mapping_filtered.csv'))

In [18]:
gd_df = gd_df.merge(mapping_df.drop(['GD_CompanyName', 'similarity'], axis=1), on='GD_CompanyID', how='left')
gd_df.to_pickle(os.path.join(const.TEMP_PATH, '2008_2023_glassdoor_reviews_with_gvkey.pkl'))

In [21]:
gd_df['GD_ReviewDate'] = pd.to_datetime(gd_df['GD_ReviewDate'])
gd_df[const.YEAR] = gd_df['GD_ReviewDate'].dt.year

In [4]:
ctat_df = pd.read_csv(os.path.join(const.COMPUSTAT_PATH, '1950_2024_ctat_firm_identifiers.zip'))
ctat_df['datadate'] = pd.to_datetime(ctat_df['datadate'])
ctat_df[const.YEAR] = ctat_df['fyear'].fillna(ctat_df['datadate'].apply(lambda x: x.year if x.month >= 6 else x.year - 1))
ctat_df.drop_duplicates(subset=[const.GVKEY, const.YEAR], keep='last', inplace=True)

C:\Users\wangy\AppData\Local\Temp\ipykernel_9344\2800178292.py:1: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  ctat_df = pd.read_csv(os.path.join(const.COMPUSTAT_PATH, '1950_2024_ctat_firm_identifiers.zip'))


In [5]:
ctat_df = ctat_df[ctat_df[const.YEAR] >= 2008].copy()
ctat_df.head()

,costat,curcd,datafmt,indfmt,consol,tic,datadate,gvkey,conm,cusip,...,city,conml,ipodate,loc,phone,sic,state,weburl,fyear,year
96,A,USD,STD,INDL,C,AIR,2009-05-31,1004,AAR CORP,000361105,...,Wood Dale,AAR Corp,1972-04-24,USA,630 227 2000,5080,IL,www.aarcorp.com,2008,2008
97,A,USD,STD,INDL,C,AIR,2010-05-31,1004,AAR CORP,000361105,...,Wood Dale,AAR Corp,1972-04-24,USA,630 227 2000,5080,IL,www.aarcorp.com,2009,2009
99,A,USD,STD,INDL,C,AIR,2011-05-31,1004,AAR CORP,000361105,...,Wood Dale,AAR Corp,1972-04-24,USA,630 227 2000,5080,IL,www.aarcorp.com,2010,2010
100,A,USD,STD,INDL,C,AIR,2012-05-31,1004,AAR CORP,000361105,...,Wood Dale,AAR Corp,1972-04-24,USA,630 227 2000,5080,IL,www.aarcorp.com,2011,2011
101,A,USD,STD,INDL,C,AIR,2013-05-31,1004,AAR CORP,000361105,...,Wood Dale,AAR Corp,1972-04-24,USA,630 227 2000,5080,IL,www.aarcorp.com,2012,2012


In [28]:
gd_df_firm_year = gd_df[['GD_CompanyID', 'GD_CompanyLink', 'GD_CompanyName', const.YEAR, const.GVKEY]].drop_duplicates(
    subset=['GD_CompanyID', const.YEAR], keep='last')
merged_df = gd_df_firm_year.merge(ctat_df[[const.GVKEY, const.YEAR, 'conm', 'conml']], how='left', on=[const.GVKEY, const.YEAR])

In [29]:
merged_df.shape

(223433, 7)

In [34]:
merged_df[merged_df['conm'].notnull() & merged_df[const.GVKEY].notnull()].shape

(20875, 7)

In [38]:
merged_df.loc[merged_df['conm'].isna() & merged_df[const.GVKEY].notnull(), const.GVKEY] = np.nan

In [41]:
merged_df.to_pickle(os.path.join(const.TEMP_PATH, '2008_2023_glassdoor_names_ids.pkl'))

In [39]:
print(ctat_df.head().to_csv(index=False))

costat,curcd,datafmt,indfmt,consol,tic,datadate,gvkey,conm,cusip,cik,add1,add2,addzip,city,conml,ipodate,loc,phone,sic,state,weburl,fyear,year
A,USD,STD,INDL,C,AIR,2009-05-31,1004,AAR CORP,000361105,1750.0,"One AAR Place, 1100 North Wood Dale Road",,60191,Wood Dale,AAR Corp,1972-04-24,USA,630 227 2000,5080,IL,www.aarcorp.com,2008,2008
A,USD,STD,INDL,C,AIR,2010-05-31,1004,AAR CORP,000361105,1750.0,"One AAR Place, 1100 North Wood Dale Road",,60191,Wood Dale,AAR Corp,1972-04-24,USA,630 227 2000,5080,IL,www.aarcorp.com,2009,2009
A,USD,STD,INDL,C,AIR,2011-05-31,1004,AAR CORP,000361105,1750.0,"One AAR Place, 1100 North Wood Dale Road",,60191,Wood Dale,AAR Corp,1972-04-24,USA,630 227 2000,5080,IL,www.aarcorp.com,2010,2010
A,USD,STD,INDL,C,AIR,2012-05-31,1004,AAR CORP,000361105,1750.0,"One AAR Place, 1100 North Wood Dale Road",,60191,Wood Dale,AAR Corp,1972-04-24,USA,630 227 2000,5080,IL,www.aarcorp.com,2011,2011
A,USD,STD,INDL,C,AIR,2013-05-31,1004,AAR CORP,000361105,1750.0,"One AAR Place, 11

In [6]:
merged_df = pd.read_pickle(os.path.join(const.TEMP_PATH, '2008_2023_glassdoor_names_ids.pkl'))

In [8]:
def _default_preprocess(name: str) -> str:
    """
    Conservative normalization for company names.
    Keep it light—embedding models already handle a lot.
    """
    if name is None or (isinstance(name, float) and np.isnan(name)):
        return ""
    s = str(name).strip().lower()
    s = re.sub(r"\s+", " ", s)
    # Optional: remove some punctuation that often varies across sources
    s = re.sub(r"[^\w\s&.-]", "", s)
    return s.strip()


def _parse_find_matches_output(matches):
    """
    company-name-matcher's find_matches() returns a list of matches;
    handle common shapes defensively:
      - [("Apple Inc", 0.91), ...]
      - [{"match": "Apple Inc", "score": 0.91}, ...]
      - [{"company": "Apple Inc", "similarity": 0.91}, ...]
      - ["Apple Inc", ...]  (rare)
    Returns: (best_name, best_score) or (None, None)
    """
    if not matches:
        return None, None

    m0 = matches[0]

    # tuple/list: (name, score)
    if isinstance(m0, (tuple, list)) and len(m0) >= 2:
        return m0[0], float(m0[1])

    # dict-like
    if isinstance(m0, dict):
        name = m0.get("match") or m0.get("company") or m0.get("name")
        score = m0.get("score") or m0.get("similarity")
        try:
            score = float(score) if score is not None else None
        except Exception:
            score = None
        return name, score

    # plain string
    if isinstance(m0, str):
        return m0, None

    return None, None


def fuzzy_fill_gvkey_conm_conml(
    merged_df: pd.DataFrame,
    ctat_df: pd.DataFrame,
    *,
    merged_name_col: str = "GD_CompanyName",
    merged_year_col: str = "year",
    ctat_year_col: str = "year",
    ctat_name_col: str = "conml",          # use conml for matching per your rule
    fallback_ctat_name_col: str = "conm",  # if conml missing
    threshold: float = 0.80,
    k: int = 1,
    use_approx: bool = False,
    n_clusters: int = 50,
    model_name: str = "paraphrase-multilingual-MiniLM-L12-v2",
) -> pd.DataFrame:
    """
    Rules implemented:
      1) If gvkey is not null -> skip row
      2) If gvkey is null -> fuzzy match using Compustat name (ctat_df conml; fallback conm)
      3) Only match within the same year
      4) If matched -> fill (gvkey, conm, conml) from ctat_df

    Notes:
      - Builds a separate vector index per year (fast enough for typical sizes).
      - If your ctat_df is huge, consider setting use_approx=True and tune n_clusters.
    """
    out = merged_df.copy()

    # Ensure year is integer for reliable grouping/joining
    out[merged_year_col] = pd.to_numeric(out[merged_year_col], errors="coerce").astype("Int64")
    ctat = ctat_df.copy()
    ctat[ctat_year_col] = pd.to_numeric(ctat[ctat_year_col], errors="coerce").astype("Int64")

    # Build a matcher (preprocess_fn keeps names consistent across both sides)
    matcher = CompanyNameMatcher(model_name, preprocess_fn=_default_preprocess)

    # Work only on rows where gvkey is missing
    missing_mask = out["gvkey"].isna()
    if missing_mask.sum() == 0:
        return out

    # Precompute a per-year Compustat candidate table
    # Use conml for matching; fallback to conm if conml missing
    ctat["_match_name_raw"] = ctat[ctat_name_col]
    ctat.loc[ctat["_match_name_raw"].isna(), "_match_name_raw"] = ctat[fallback_ctat_name_col]
    ctat["_match_name_raw"] = ctat["_match_name_raw"].astype(str)

    # For lookup after matching: (year, normalized_name) -> row (gvkey, conm, conml)
    # If duplicates exist, keep first occurrence.
    ctat["_match_name_norm"] = ctat["_match_name_raw"].map(_default_preprocess)
    ctat_lookup = (
        ctat.dropna(subset=[ctat_year_col])
            .drop_duplicates(subset=[ctat_year_col, "_match_name_norm"])
            .set_index([ctat_year_col, "_match_name_norm"])[["gvkey", "conm", "conml"]]
    )

    # Iterate by year for year-restricted matching
    years_to_process = out.loc[missing_mask, merged_year_col].dropna().unique().tolist()

    for y in years_to_process:
        # Candidate Compustat names for this year
        ctat_y = ctat.loc[ctat[ctat_year_col] == y, ["_match_name_raw", "_match_name_norm"]].dropna()
        if ctat_y.empty:
            continue

        candidates_raw = ctat_y["_match_name_raw"].tolist()

        # Build an index for this year
        # If use_approx=True, n_clusters should scale with number of candidates.
        matcher.build_index(
            candidates_raw,
            n_clusters=min(n_clusters, max(2, int(len(candidates_raw) ** 0.5))) if use_approx else 20,
            save_dir=None,
        )

        # Rows in merged_df for this year needing fill
        idxs = out.index[missing_mask & (out[merged_year_col] == y)]
        if len(idxs) == 0:
            continue

        for idx in idxs:
            query_name = out.at[idx, merged_name_col]
            if pd.isna(query_name) or str(query_name).strip() == "":
                continue

            matches = matcher.find_matches(
                query_name,
                threshold=threshold,
                k=k,
                use_approx=use_approx,
            )

            best_name, best_score = _parse_find_matches_output(matches)
            if not best_name:
                continue

            # Lookup by (year, normalized matched name)
            key = (y, _default_preprocess(best_name))
            if key not in ctat_lookup.index:
                continue

            gvkey_val, conm_val, conml_val = ctat_lookup.loc[key, ["gvkey", "conm", "conml"]].tolist()

            # Fill only missing fields (per your rule: replace missing gvkey/conm/conml)
            if pd.isna(out.at[idx, "gvkey"]):
                out.at[idx, "gvkey"] = gvkey_val
            if pd.isna(out.at[idx, "conm"]) or str(out.at[idx, "conm"]).strip() == "":
                out.at[idx, "conm"] = conm_val
            if pd.isna(out.at[idx, "conml"]) or str(out.at[idx, "conml"]).strip() == "":
                out.at[idx, "conml"] = conml_val

            # Optional: store match diagnostics
            # out.at[idx, "match_score"] = best_score
            # out.at[idx, "matched_name"] = best_name

    return out

In [9]:
updated_merged_df = fuzzy_fill_gvkey_conm_conml(
    merged_df,
    ctat_df,
    threshold=0.85,   # tune
    k=1,
    use_approx=False  # set True if ctat_df per year is very large
)

d:\anaconda3\envs\GlassDoor\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\wangy\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. 

In [12]:
updated_merged_df[updated_merged_df['gvkey'].notnull()].shape

(45528, 7)

In [4]:
updated_merged_df = pd.read_pickle(os.path.join(const.TEMP_PATH, '2008_2023_glassdoor_names_ids_fuzzy_filled.pkl'))

In [5]:
firm_merged_df = updated_merged_df[updated_merged_df['gvkey'].notnull()].copy()
firm_merged_df.head()

,GD_CompanyID,GD_CompanyLink,GD_CompanyName,year,gvkey,conm,conml
134,765,Reviews/Allied-Healthcare-Products-Reviews-E76...,Allied Healthcare Products,2022,24837.0,ALLIED HEALTHCARE PRODS INC,Allied Healthcare Products Inc
135,765,Reviews/Allied-Healthcare-Products-Reviews-E76...,Allied Healthcare Products,2021,24837.0,ALLIED HEALTHCARE PRODS INC,Allied Healthcare Products Inc
136,765,Reviews/Allied-Healthcare-Products-Reviews-E76...,Allied Healthcare Products,2018,24837.0,ALLIED HEALTHCARE PRODS INC,Allied Healthcare Products Inc
137,765,Reviews/Allied-Healthcare-Products-Reviews-E76...,Allied Healthcare Products,2017,24837.0,ALLIED HEALTHCARE PRODS INC,Allied Healthcare Products Inc
138,765,Reviews/Allied-Healthcare-Products-Reviews-E76...,Allied Healthcare Products,2016,24837.0,ALLIED HEALTHCARE PRODS INC,Allied Healthcare Products Inc


In [6]:
firm_mapping = firm_merged_df[[const.GD_COMPANYID, const.YEAR, const.GVKEY]].drop_duplicates()
firm_mapping.shape

(45528, 3)

In [8]:
gd_df = pd.read_pickle(os.path.join(const.TEMP_PATH, '2008_2023_raw_glassdoor_reviews.pkl'))
gd_df.head()

,GD_JobTitle,GD_CompanyLink,GD_CompanyName,GD_CompanyID,GD_ReviewDate,GD_Rating,GD_ReviewTitle,GD_ReviewerStatus,GD_Pros,GD_Cons,...,GD_Recommend,GD_CEOSupport,GD_Outlook,GD_CareerOpp,GD_CompBenefits,GD_Management,GD_WorkLife,GD_CultureValues,GD_Diversity,GD_Index
0,Manager Design,Reviews/Baja-Steel-and-Fence-Reviews-E5462645.htm,Baja Steel and Fence,5462645,2022-11-19,5.0,Good,"Current Employee, more than 10 years",Knowledge gain of complete project,Financial growth and personal growth,...,v,o,v,3.0,3.0,3.0,3.0,3.0,3.0,NaN
1,Anonymous Employee,Reviews/Baja-Steel-and-Fence-Reviews-E5462645.htm,Baja Steel and Fence,5462645,2022-01-29,4.0,Good,"Former Employee, less than 1 year","Good work,good work , flexible, support","Good,work, flexible,good support, good team work",...,v,o,o,4.0,4.0,4.0,4.0,4.0,4.0,NaN
2,Production Engineer,Reviews/Baja-Steel-and-Fence-Reviews-E5462645.htm,Baja Steel and Fence,5462645,2021-08-12,4.0,"Supervising the manufacturing the processes, e...","Current Employee, more than 1 year",This company is a best opportunity for me to l...,"Monthly Target work,Maintain production schedu...",...,v,o,v,2.0,3.0,2.0,2.0,2.0,2.0,NaN
3,Senior Account Executive,https://www.glassdoor.com/Reviews/Calgary-Flam...,Calgary Flames,5247,2020-09-24,1.0,terrible,"Current Employee, more than 1 year",I wish there were some to list,too many to list here,...,x,x,x,1.0,3.0,1.0,3.0,1.0,NaN,NaN
4,Assistant Manager,https://www.glassdoor.com/Reviews/Calgary-Flam...,Calgary Flames,5247,2023-03-25,4.0,"It could be so good, but it isn’t","Current Employee, more than 3 years",Fast Paced. Endless challenges. Inclusive envi...,The biggest perk of the job provides no value ...,...,o,o,o,3.0,3.0,3.0,1.0,4.0,5.0,NaN


In [9]:
gd_df[const.GD_REVIEWDATE] = pd.to_datetime(gd_df[const.GD_REVIEWDATE])
gd_df[const.YEAR] = gd_df[const.GD_REVIEWDATE].dt.year
gd_df = gd_df.merge(firm_mapping, on=[const.GD_COMPANYID, const.YEAR], how='inner')
gd_df.shape

(3994240, 23)

In [38]:
gd_df.to_pickle(os.path.join(const.TEMP_PATH, '2008_2023_glassdoor_reviews_with_gvkey_v2.pkl'))

In [22]:
gd_df['month'] = gd_df[const.GD_REVIEWDATE].dt.month
gd_monthly_df = gd_df.groupby([const.GVKEY, const.YEAR, 'month']).agg(
    num_reviews=(const.GD_COMPANYNAME, 'count'),
    rating_mean=(const.GD_RATING, 'mean'),
    career_opp_mean=(const.GD_CAREEROPP, 'mean'),
    comp_benefits_mean=(const.GD_COMPBENEFITS, 'mean'),
    management_mean=(const.GD_MANAGEMENT, 'mean'),
    work_life_mean=(const.GD_WORKLIFE, 'mean'),
    culture_values_mean=(const.GD_CULTUREVALUES, 'mean'),
    diversity_mean=(const.GD_DIVERSITY, 'mean'),
    GD_CompanyName=(const.GD_COMPANYNAME, 'last'),
    GD_CompanyID=(const.GD_COMPANYID, 'last'),
).reset_index(drop=False)
gd_monthly_df.shape

(196406, 13)

In [12]:
gd_monthly_df.describe()

,gvkey,year,month,num_reviews,rating_mean,career_opp_mean,comp_benefits_mean,management_mean,work_life_mean,culture_values_mean,diversity_mean
count,196406.000000,196406.000000,196406.000000,196406.000000,196406.000000,186791.000000,184523.000000,187371.000000,185600.000000,160424.000000,41686.000000
mean,57702.326889,2016.460556,6.468901,20.336650,3.293857,3.064031,3.313728,2.908932,3.303461,3.254291,3.651292
std,65410.504447,3.972486,3.455401,92.905309,0.926562,0.940242,0.871895,1.000745,0.947993,1.008731,0.863133
min,1004.000000,2008.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25%,10385.000000,2013.000000,3.000000,1.000000,2.857143,2.500000,2.952381,2.285714,2.833333,2.692308,3.250000
50%,25466.000000,2017.000000,6.000000,4.000000,3.363636,3.000000,3.333333,3.000000,3.333333,3.333333,3.800000
75%,102523.000000,2020.000000,9.000000,11.000000,4.000000,3.666667,4.000000,3.500000,4.000000,4.000000,4.138758
max,351038.000000,2023.000000,12.000000,6264.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000


In [13]:
ctat_df = pd.read_csv(os.path.join(const.COMPUSTAT_PATH, '1950_2023_ctat_firm_ta_ctrl.zip'))
ctat_df['datadate'] = pd.to_datetime(ctat_df['datadate'])
ctat_df[const.YEAR] = ctat_df['fyear'].fillna(ctat_df['datadate'].apply(lambda x: x.year if x.month >= 6 else x.year - 1))
ctat_df.drop_duplicates(subset=[const.GVKEY, const.YEAR], keep='last', inplace=True)
ctat_df.head()

,gvkey,datadate,fyear,indfmt,consol,popsrc,datafmt,tic,cusip,conm,...,addzip,city,conml,county,incorp,loc,sic,state,ipodate,year
0,1000,1961-12-31,1961.0,INDL,C,D,STD,AE.2,000032102,A & E PLASTIK PAK INC,...,NaN,NaN,A & E Plastik Pak Inc,NaN,NaN,USA,3089,NaN,NaN,1961.0
1,1000,1962-12-31,1962.0,INDL,C,D,STD,AE.2,000032102,A & E PLASTIK PAK INC,...,NaN,NaN,A & E Plastik Pak Inc,NaN,NaN,USA,3089,NaN,NaN,1962.0
2,1000,1963-12-31,1963.0,INDL,C,D,STD,AE.2,000032102,A & E PLASTIK PAK INC,...,NaN,NaN,A & E Plastik Pak Inc,NaN,NaN,USA,3089,NaN,NaN,1963.0
3,1000,1964-12-31,1964.0,INDL,C,D,STD,AE.2,000032102,A & E PLASTIK PAK INC,...,NaN,NaN,A & E Plastik Pak Inc,NaN,NaN,USA,3089,NaN,NaN,1964.0
4,1000,1965-12-31,1965.0,INDL,C,D,STD,AE.2,000032102,A & E PLASTIK PAK INC,...,NaN,NaN,A & E Plastik Pak Inc,NaN,NaN,USA,3089,NaN,NaN,1965.0


In [24]:
ctat_df2 = ctat_df[[const.GVKEY, const.YEAR, 'at', 'ceq', 'csho', 'dlc', 'dltt',
       'intan', 'pi', 'pifo', 'ppent', 'seq', 'spi', 'tlcf', 'txdfed', 'txdfo',
       'txdi', 'txpd', 'txr', 'txt', 'xrd', 'xsga', 'sic', 'state', 'ipodate', 'conml']].copy()
gd_monthly_df2 = gd_monthly_df.merge(ctat_df2, on=[const.GVKEY, const.YEAR], how='inner')
gd_monthly_df2.shape

(190388, 37)

In [34]:
gd_monthly_df2.loc[gd_monthly_df2[const.GD_REVIEWDATE].apply(lambda x: x > pd.Timestamp('2017-10-31') and x < pd.Timestamp('2019-05-01'))].shape

KeyError: 'GD_ReviewDate'

In [28]:
matched_df = gd_monthly_df2[[const.GD_COMPANYNAME, 'conml', const.GD_COMPANYID, const.GVKEY, const.YEAR]].drop_duplicates()

In [29]:
matched_df.head()

,GD_CompanyName,conml,GD_CompanyID,gvkey,year
0,AAR,AAR Corp,4,1004.0,2008.0
1,AAR,AAR Corp,4,1004.0,2009.0
2,AAR,AAR Corp,4,1004.0,2010.0
3,AAR,AAR Corp,4,1004.0,2011.0
8,AAR,AAR Corp,4,1004.0,2012.0


In [32]:

# ---------- preprocessing ----------
def _preprocess(name):
    if name is None or (isinstance(name, float) and np.isnan(name)):
        return ""
    s = str(name).lower().strip()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^\w\s&.-]", "", s)
    return s


# ---------- matcher ----------
matcher = CompanyNameMatcher(
    "paraphrase-multilingual-MiniLM-L12-v2",
    preprocess_fn=_preprocess
)


def add_pairwise_match_score(
    df: pd.DataFrame,
    left_col: str = "GD_CompanyName",
    right_col: str = "conml",
    year_col: str = "year",
) -> pd.DataFrame:
    """
    Add `match_score` measuring fuzzy similarity between left_col and right_col.
    Uses company-name-matcher correctly via find_matches().
    """
    out = df.copy()
    out["match_score"] = np.nan

    years = out[year_col].dropna().unique()

    for y in tqdm(years, desc="Scoring by year"):
        idxs = out.index[out[year_col] == y]

        for idx in tqdm(idxs, desc=f"Year {int(y)}", leave=False):
            left = out.at[idx, left_col]
            right = out.at[idx, right_col]

            if pd.isna(left) or pd.isna(right):
                continue

            # build a temporary index with ONLY the matched name
            matcher.build_index([right], save_dir=None)

            matches = matcher.find_matches(left, k=1, threshold=0.0)

            if matches:
                # matches[0] is typically (name, score)
                score = matches[0][1] if isinstance(matches[0], (tuple, list)) else np.nan
                out.at[idx, "match_score"] = score

    return out

In [33]:
matched_df2 = add_pairwise_match_score(matched_df)


Scoring by year:   0%|          | 0/16 [00:00<?, ?it/s]

Year 2008:   0%|          | 0/1295 [00:00<?, ?it/s]

Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans 

Year 2009:   0%|          | 0/1390 [00:00<?, ?it/s]

Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans 

...

Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans 

Year 2022:   0%|          | 0/2126 [00:00<?, ?it/s]

Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans 

Year 2023:   0%|          | 0/60 [00:00<?, ?it/s]

Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans index with 1 or 0 items. Skipping index creation.
Cannot build KMeans 

In [36]:
# Count observations in gd_monthly_df2 between 2017-11 and 2019-04 (inclusive)
# Uses year/month since monthly aggregation doesn't have GD_ReviewDate.
mask = (
    ((gd_monthly_df2[const.YEAR] == 2017) & (gd_monthly_df2['month'] >= 11)) |
    (gd_monthly_df2[const.YEAR] == 2018) |
    ((gd_monthly_df2[const.YEAR] == 2019) & (gd_monthly_df2['month'] <= 4))
)
obs_count = int(mask.sum())
print(f"Observations between 2017-11 and 2019-04: {obs_count}")
# Optional: show the filtered shape directly
gd_monthly_df2.loc[mask].shape

Observations between 2017-11 and 2019-04: 22620


(22620, 37)

In [47]:
gd_monthly_df2.to_pickle(os.path.join(const.TEMP_PATH, 'glassdoor_monthly_compustat_ann_2008_2023.pkl'))